# Analysis

## Setting parameters

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from hhelper.tester import calculate_score, count_all_rules, count_all_attributes
from hhelper.typesetting import bold
from pathlib import Path
from hhelper.scoring import balanced_accuracy_score
import pandas as pd
import numpy as np

In [159]:
data_sets = [  # actual set for QR
    'australian',
     # 'automobile', # cat feats
     'bands',
     'bupa',
     'cleveland',
     # 'contraceptive', # no rules found
     # 'crx', # cat feats
     'dermatology',
     'ecoli',
     # 'german', # cat feats
     'glass', 
     # 'haberman', # no rules found
     'heart',
     'ionosphere',
     # 'mammographic',  # no rules found
     'pima',
     # 'saheart', # cat feats
     'sonar',  # 70 features!
     'spectfheart',
     'vehicle',
     'vowel',
     'wine',
     'winequality-red',  # todo temp uit voor inclusion tests
     'wisconsin',
     'yeast'
]

def_sets = [
    'australian',
    'bupa',
    'cleveland',
    # 'crx',
    # 'dermatology',
    'ecoli',
    'glass',
    # 'haberman',
    'heart',
    'ionosphere',
    'pima',
    # 'sonar',
    'spectfheart',
    # 'saheart',
    # 'wdbc',
    'wine',
    'winequality-red',
    'wisconsin',
    'yeast',
    # 'automobile',
    # 'dermatology', run base again
    # 'german',
    # 'movement',
    # 'vehicle',
    # 'vowel',
]
inclusion_test = [
    # 'ecoli',
    # 'wisconsin',
    # 'dermatology',
    # 'cleveland',
    # 'spectfheart',
    # 'bupa',
    # 'yeast',
    # 'heart',
    # 'australian',
    # 'glass',
    # 'pima',
    'australian',
     'bands',
     'bupa',
     'cleveland',
     # 'dermatology', temp 01/07/24
     'ecoli',
     'glass',   # todo tijdelijk bij prior-incl
     'heart',
     'ionosphere',
     'pima',
     'sonar',  # 70 features! eventjes skip voor QuickReduct 25/04/2024
     'spectfheart',
     'vehicle',
     # 'vowel',
     'wine',
     # 'winequality-red',  # temp uit voor inclusion tests
     'wisconsin',
     'yeast'  # todo tijdelijk bij prior-incl
]

qr_list = [
    'australian',
    'bupa',
    'cleveland',
    'dermatology',
    'ecoli',
    'glass',
    'heart',
    'pima',
    'spectfheart',
    # 'vehicle',
    # 'vowel',
    'wisconsin',
    'yeast'
]

In [41]:
# data_sets = inclusion_test

In [42]:
len(data_sets)

17

In [43]:
data_base = Path.cwd() / 'keel-data'
metric = balanced_accuracy_score  # balanced_
scores = {}
rules = {}
attributes = {}
results_base = Path.cwd() / 'results'

## Loading WEKA results
MODLEM, FURIA, RIPPER

In [44]:
weka_folder = Path.cwd() / 'weka-results'
weka_models = {
    'modlem':'&',
    'furia':'and',
    'ripper':'and'
}
acc_type = 'balanced-accuracy'  # 'weka-accuracy.csv'

In [45]:
for name, connection in weka_models.items():
    model_folder = weka_folder / f"{name}-files"
    file = model_folder / f"{name}-{acc_type}.csv"
    scores[name] = pd.read_csv(weka_folder / file, header=None, index_col=0).to_dict()[1]
    rules[name] = {}
    attributes[name] = {}
    for file in model_folder.iterdir():
        if file.name[-3:] == 'txt':
            short_name = file.name[:-4]
            with open(file, 'r') as f:
                line = f.readline()
                nrs = []
                while len(line) > 4:
                    nrs.append(line.count(connection) + 1)
                    line = f.readline()
            rules[name][short_name] = len(nrs)
            attributes[name][short_name] = np.average(nrs)

## Loading QuickRules results

In [46]:
qr_models = {
    'qr': results_base / 'no-prune-results' / 'close-min-max-combo-results',
    # 'lin-owa': results_base / 'no-prune-results' / 'rules-lin-owa-qr-combo-minmax-results',
    # 'trun-lin-owa': results_base / 'no-prune-results' / 'rules-trun-lin-owa-qr-combo-minmax-results',
    # 'trun-exp-owa': results_base / 'no-prune-results' / 'trun-exp-owa-qr-combo-minmax-results',
    # 'avg-qr': results_base / 'mon-avg-std-minmax-results-2',
    # 'avg-lin-owa' : results_base / 'mon-avg-lin-owa-minmax-results-2',
    # 'prune-qr': results_base / 'prune-results' / 'prune-qr-combo-minmax-results',
    # 'prune-lin-owa': results_base / 'prune-results' / 'prune-lin-owa-qr-combo-minmax-results',
    # 'prune-trun-lin-owa': results_base / 'prune-results' / 'prune-trun-lin-owa-qr-combo-min-max-results',
    # 'prune-trun-exp-owa': results_base / 'prune-results' / 'prune-trun-exp-owa-qr-combo-min-max-results',
    # 'prune-avg-qr': results_base / 'prune-mon-avg-std-minmax-results-2',
    # 'prune-avg-lin-owa' : results_base / 'prune-mon-avg-lin-owa-minmax-results-2',
}

In [47]:
for model, path in qr_models.items():
    scores[model] = calculate_score(
        data_folder=data_base,
        results_folder=path,
        include=data_sets,
        metric=metric
    )
    rules[model] = count_all_rules(
        Path.cwd() / results_base / path,
        include=data_sets
    )
    attributes[model] = count_all_attributes(
        Path.cwd() / results_base / path,
        include=data_sets
    )

## Adding results for non rule based models


In [48]:
other_models =  {
    'naive-bayes': Path.cwd() / 'NaiveBayes-results',
    'svm': Path.cwd() / 'svm',
    'tree': Path.cwd() / 'decision-tree'
}

In [49]:
for model, path in other_models.items():
    print(model)
    scores[model] = calculate_score(
        data_folder=data_base,
        results_folder=path,
        include=data_sets,
        metric=metric,
        verbose=False
    )

naive-bayes
svm
tree


In [50]:
frnn_results = pd.read_csv(Path.cwd() / 'frnn_new_results.csv', header=None, index_col=0).to_dict()[1]
scores['frnn'] =  {data_set: frnn_results[data_set] for data_set in data_sets}

## FRRI Models

In [249]:
frri_models = {
    'incl 1e-4' : Path.cwd() / results_base / 'lower-approx-new-impl-test',
    'frfs': Path.cwd() / results_base / 'lower-approx' / 'frfs',
    # 'msecvx-i50-reducts': Path.cwd() / 'results' / 'multiclassMSECVX' / 'reducts-incl5-cov05',
    # 'relabel-msecvx-reducts-default': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relabelling-reducts-default',
    # 'relabel-maecvx-reducts-default': Path.cwd() / 'results' / 'multiclassMAECVX' / 'relabelling-reducts-default',
    # 'relabel-msecvx-c50-reducts': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relabelling-reducts-cov50',
    # 'relabel-maecvx-c50-reducts': Path.cwd() / 'results' / 'multiclassMAECVX' / 'relabelling-reducts-cov50',
    # 'relabel-msecvx-i50-reducts': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relabelling-reducts-in50',
    # 'relabel-maecvx-i50-reducts': Path.cwd() / 'results' / 'multiclassMAECVX' / 'relabelling-reducts-in50',
    # 'relabel-msecvx-i95-reducts': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relabelling-reducts-in95',
    # 'msecvx-relab-red-cov50-i50-cer51': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab-red-cov50-inc95-cer51',
    # 'msecvx-relab-red-cov50-i50-cer55': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab-red-cov50-inc95-cer55',
    # 'msecvx-relab-red-cov50-i50-cer60': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab-red-cov50-inc95-cer60',
    # 'msecvx-relab-red-cov50-i50-cer70': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab-red-cov50-inc95-cer70',
    # 'msecvx-relab-red-cov50-i50-cer51-na': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab-red-cov50-inc95-cer51-no_add',
    # 'msecvx-relab-red-cov50-i50-cer55-na': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab-red-cov50-inc95-cer55-no_add',
    # 'msecvx-relab-red-cov50-i50-cer60-na': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab-red-cov50-inc95-cer60-no_add',
    # 'msecvx-relab-red-cov50-i50-cer70-na': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab-red-cov50-inc95-cer70-no_add',
    # 'msecvx-relab02-red-cov50-i50': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab02-reducts-cov50-inc95',
    # 'msecvx-relab05-red-cov50-i50': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab05-reducts-cov50-inc95',
    # 'msecvx-relab10-red-cov50-i50': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab10-reducts-cov50-inc95',
    # 'msecvx-relab15-red-cov50-i50': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab15-red-cov50-inc95',
    # 'relabel-maecvx-i95-reducts': Path.cwd() / 'results' / 'multiclassMAECVX' / 'relabelling-reducts-in95',
    # 'relabel-msecvx-i50+c50-reducts': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relabelling-reducts-incl5-cov05-v2',
    # 'relabel-maecvx-i50+c50-reducts': Path.cwd() / 'results' / 'multiclassMAECVX' / 'relabelling-reducts-cov50-inc50',
    # 'relabel-msecvx-i95+c50-reducts': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relabelling-reducts-in95-cov50',
    # 'relabel-maecvx-i95+c50-reducts': Path.cwd() / 'results' / 'multiclassMAECVX' / 'relabelling-reducts-cov50-inc95',
    # 'default-check': Path.cwd() / 'results' / 'default',
    # 'msecvx-relab05-red-cov25-i50-cer55-na': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab05-red-cov25-inc50-cer55-no_add',
    # 'msecvx-relab05-red-cov25-i95-cer55-na': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab05-red-cov25-inc95-cer55-no_add',
    # 'msecvx-relab05-red-cov50-i50-cer55-na': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab05-red-cov50-inc50-cer55-no_add',
    # 'msecvx-relab05-red-cov50-i95-cer55-na': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab05-red-cov50-inc95-cer55-no_add',
    # 'msecvx-relab05-red-cov50-ie-6-cer55-na': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab05-red-cov50-inc-e-6-cer55-no_add',
    # 'msecvx-relab05-red-cov55-ie-6-cer55-na': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab05-red-cov55-inc-e-6-cer55-no_add',
    # 'msecvx-relab05-red-cov50-i55-cer55-na': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab05-red-cov50-inc55-cer55-no_add',
    # 'msecvx-relab05-red-cov50-i95-cer70-na': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab05-red-cov50-inc95-cer70-no_add',
    # 'msecvx-relab-red-cov50-i55-cer55-na': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab-red-cov50-inc55-cer55-no_add',
    # 'msecvx-relab-red-cov50-i45-cer55-na': Path.cwd() / 'results' / 'multiclassMSECVX' / 'relab-red-cov50-inc45-cer55-no_add',
    # 'gran-mse-relab-red-cov50-inc50': Path.cwd() / 'results' / 'multiclassMSECVXgran' / 'relab-red-cov50-inc50',
    # 'gran-mse-relab-red-cov50': Path.cwd() / 'results' / 'multiclassMSECVXgran' / 'relab-red-cov50',
    # 'gran-mse-relab-red-inc50': Path.cwd() / 'results' / 'multiclassMSECVXgran' / 'relab-red-i50',
    # 'gran-mse-relab-red': Path.cwd() / 'results' / 'multiclassMSECVXgran' / 'relab-red',
    'control': Path.cwd() / 'results' / 'multiclassMSECVXgran' / 'control-does-the-old-work',
    # 'ijcrs-frfs': Path.cwd() / 'results' / 'ijcrs-retest' / 'frfs-true',
    'ofrfs-1' : Path.cwd() / 'results' / 'ijcrs-retest' / 'ofrfs',
    'ofrfs-0.9': Path.cwd() / 'results' / 'ijcrs-retest' / 'ofrfs-0.9',
    'ofrfs-0.8': Path.cwd() / 'results' / 'ijcrs-retest' / 'ofrfs-0.8',
    'ofrfs-0' : Path.cwd() / 'results' / 'ijcrs-retest' / 'ofrfs-0',
    'mi-1' : Path.cwd() / 'results' / 'ijcrs-retest' / 'mi-1',
    'mi-0.9': Path.cwd() / 'results' / 'ijcrs-retest' / 'mi-0.9',
    'mi-0.8': Path.cwd() / 'results' / 'ijcrs-retest' / 'mi-0.8',
    'pcc-1' : Path.cwd() / 'results' / 'ijcrs-retest' / 'pcc-1',
    'pcc-0.9': Path.cwd() / 'results' / 'ijcrs-retest' / 'pcc-0.9',
    'pcc-0.8': Path.cwd() / 'results' / 'ijcrs-retest' / 'pcc-0.8',
}
old_models = {
    # 'prio 1e-6' : Path.cwd() / 'results' / 'lower-approx' / 'prior-inclusion' / '1e-6',
    # 'prio-max 1e-6' : Path.cwd() / 'results' / 'lower-approx' / 'prior-inclusion' / '1e-6-scale',
    # '1e-6-high' : Path.cwd() / 'results' / 'lower-approx' / 'prior-inclusion' / '1e-6-high',
    # '1e-6-high-sum' : Path.cwd() / 'results' / 'lower-approx' / 'prior-inclusion' / '1e-6-high-sum',
    # '1e-6-trueprior-influence-1e-2': Path.cwd() / 'results' / 'lower-approx' / 'prior-inclusion' / '1e-6-trueprior-influence-1e-2',
    # '1e-6-trueprior-influence-1e-3' : Path.cwd() / 'results' / 'lower-approx' / 'prior-inclusion' / '1e-6-trueprior-influence-1e-3',
    # '1e-6-trueprior-influence-1e-4' : Path.cwd() / 'results' / 'lower-approx' / 'prior-inclusion' / '1e-6-trueprior-influence-1e-4',
    # '1e-6-trueprior-influence-1e-6' : Path.cwd() / 'results' / 'lower-approx' / 'prior-inclusion' / '1e-6-trueprior-influence-1e-6',
    # '1e-6-scaled_prior-influence-1e-2' : Path.cwd() / 'results' / 'lower-approx' / 'prior-inclusion' / '1e-6-scaled_prior-influence-1e-2',
    # '1e-6-scaled_prior-influence-3e-2': Path.cwd() / 'results' / 'lower-approx' / 'prior-inclusion' / '1e-6-scaled_prior-influence-3e-2',
    # 'base-no-reducts': Path.cwd() / results_base / 'inclusion' / 'no-reducts',
    # 'owa-lower-base': Path.cwd() / results_base / 'owa-lower' / 'default',
    # 'non-overlap': Path.cwd() / 'non-overlap-rules',
    # 'nori' : Path.cwd() / 'non-overlap-rules-def_sets',
    # 'lower-nori' : Path.cwd() / 'lower-approx-rules-def_sets',
    # 'lower-check' : Path.cwd() / 'lower-approx-og-implement-test',
    # 'lower-new-check' : Path.cwd() / results_base / 'lower-approx-new-impl-test',
    # 'lower-incl-t1': Path.cwd() / results_base / 'lower-approx-luka-impl-incl.99'
    # 'incl 1e-6' : Path.cwd() / results_base / 'lower-approx' / 'inclusion' / "1-1e-6",
    # 'incl 1e-3' : Path.cwd() / results_base / 'lower-approx' / 'inclusion' / "999",
    # 'incl 1e-2' : Path.cwd() / results_base / 'lower-approx' / 'inclusion' / "99",
    # 'incl 0.95' : Path.cwd() / results_base / 'lower-approx' / 'inclusion' / "95",
    # 'owa 1e-6' : Path.cwd() / results_base / 'lower-approx' / 'linear-owa-inclusion' / '1-1e-6',
    # 'quickreduct 1e-6': Path.cwd() / results_base / 'lower-approx' / 'quickreduct-ordering-bug' / '1-1e-6',
    # 'cv-thres': Path.cwd() / results_base / 'lower-approx' / 'cv-inclusion',
    # 'msecvx-no-reducts': Path.cwd() / results_base / 'multiclassMSECVX' / 'no-reducts',
    # 'maecvx-no-reducts': Path.cwd() / results_base / 'multiclassMAECVX' / 'no-reducts',
    # 'msecvx-0.5' : Path.cwd() / 'results' / 'multiclassMSECVX' / 'no-reducts-0.5-threshold',
    # 'maecvx-0.5-reducts' : Path.cwd() / 'results' / 'multiclassMAECVX' / 'reducts-0.5-threshold',
    # 'msecvx-0.5-reducts': Path.cwd() / 'results' / 'multiclassMSECVX' / 'reducts-0.5-threshold',
    # 'msecvx-i95-reducts': Path.cwd() / 'results' / 'multiclassMSECVX' / 'reducts-incl95-cov05',
    # 'ijcrs-frfs-1' : Path.cwd() / 'results' / 'ijcrs-retest' / 'frfs-1',
    # 'ijcrs-frfs-0.9': Path.cwd() / 'results' / 'ijcrs-retest' / 'frfs-0.9',
    # 'ijcrs-frfs-0.8': Path.cwd() / 'results' / 'ijcrs-retest' / 'frfs-0.8',
    # 'ijcrs-frfs-0' : Path.cwd() / 'results' / 'ijcrs-retest' / 'frfs-0',
}

In [261]:
for model, path in frri_models.items():
    print(model)
    scores[model] = calculate_score(
        data_folder=Path.cwd() / 'keel-data',
        results_folder=path, #'rule_prune-rel-0dot00-std',  #  'rel-0dot05-std', #' no-prune-results'/ 'close-min-max-combo-results',  # /
        metric=metric,
        include=data_sets,
        label_encoding=True
    )
    rules[model] = count_all_rules(
        path,
        include=data_sets
    )
    attributes[model] = count_all_attributes(
        path,
        include=data_sets,
        counter='attribute'
    )

incl 1e-4
frfs
control
ofrfs-1
ofrfs-0.9
ofrfs-0.8
ofrfs-0
mi-1
mi-0.9
mi-0.8
pcc-1
pcc-0.9
pcc-0.8


## Checking all of the models

In [262]:
scores.keys()

dict_keys(['modlem', 'furia', 'ripper', 'qr', 'naive-bayes', 'svm', 'tree', 'frnn', 'incl 1e-4', 'frfs', 'msecvx-i50-reducts', 'relabel-msecvx-reducts-default', 'relabel-maecvx-reducts-default', 'relabel-msecvx-c50-reducts', 'relabel-maecvx-c50-reducts', 'relabel-msecvx-i50-reducts', 'relabel-maecvx-i50-reducts', 'relabel-msecvx-i95-reducts', 'msecvx-relab-red-cov50-i50-cer51', 'msecvx-relab-red-cov50-i50-cer55', 'msecvx-relab-red-cov50-i50-cer60', 'msecvx-relab-red-cov50-i50-cer70', 'msecvx-relab-red-cov50-i50-cer51-na', 'msecvx-relab-red-cov50-i50-cer55-na', 'msecvx-relab-red-cov50-i50-cer60-na', 'msecvx-relab-red-cov50-i50-cer70-na', 'msecvx-relab02-red-cov50-i50', 'msecvx-relab05-red-cov50-i50', 'msecvx-relab10-red-cov50-i50', 'msecvx-relab15-red-cov50-i50', 'relabel-maecvx-i95-reducts', 'relabel-msecvx-i50+c50-reducts', 'relabel-maecvx-i50+c50-reducts', 'relabel-msecvx-i95+c50-reducts', 'relabel-maecvx-i95+c50-reducts', 'default-check', 'msecvx-relab05-red-cov25-i50-cer55-na', 'ms

## Tables
set 1 = QR + OWA-variants without pruning
set 2 = focus on pruning

In [263]:
names = {
    1: [
        'qr',
        'lin-owa',
        'trun-lin-owa',
        'trun-exp-owa',
        'modlem'
    ],
    2: [
        'qr',
        'lin-owa',
        'prune-qr',
        'prune-lin-owa'
    ],
    3: [
        'qr',
        'lin-owa',
        'avg-qr',
        'avg-lin-owa'
    ],
    4: [
        'lin-owa',
        'modlem'
    ],
    5: [
        'lin-owa',
        'frnn',
        'svm',
        'naive-bayes',
        'tree',
    ],
    'frri-paper': [
        # 'nori',
        # 'lower-nori',
        # 'lower-check',
        'lower-new-check',
        # 'lower-incl-t1',
        'modlem',
        'qr',
        'furia',
        'ripper',
    ],
    'inclusion' : [
        'incl 1e-6',
        'incl 1e-4',
        # 'incl 1e-3', 
        # 'incl 1e-2', 
        # 'incl 0.95',
        # 'owa 1e-6',
        # 'quickreduct 1e-6',
        'cv-thres',
        # 'frfs',
        # 'prio 1e-6',
        # 'prio-max 1e-6',
        '1e-6-high',
        # '1e-6-high-sum',
        # '1e-6-trueprior-influence-1e-2',
        # '1e-6-trueprior-influence-1e-4',
        '1e-6-scaled_prior-influence-1e-2',
        '1e-6-scaled_prior-influence-3e-2',
    ],
    'quickreduct' : [
        'incl 1e-6',
        'quickreduct 1e-6',
        'frfs'
    ],
    'ijcrs': [
        'control',
        'ofrfs-1',
        'ofrfs-0.9',
        'ofrfs-0.8',
        'ofrfs-0',
        'mi-1',
        'mi-0.9',
        'mi-0.8',
        'pcc-1',
        'pcc-0.9',
        'pcc-0.8',
    ],
    'gran' : [
        'base-no-reducts',
        'msecvx-no-reducts',
        'maecvx-no-reducts',
        'msecvx-0.5',
    ],
    'gran-reducts': [
        'incl 1e-4',
        # 'msecvx-0.5-reducts',
        # 'maecvx-0.5-reducts',
        # 'msecvx-i95-reducts',
        'msecvx-i50-reducts',
        'relabel-maecvx-reducts-default',
        'relabel-maecvx-c50-reducts',
        'relabel-maecvx-i50-reducts',
        'relabel-maecvx-i95-reducts',
        'relabel-maecvx-i50+c50-reducts',
        'relabel-maecvx-i95+c50-reducts',
        'relabel-msecvx-reducts-default',
        'relabel-msecvx-c50-reducts',
        'relabel-msecvx-i50-reducts',
        'relabel-msecvx-i95-reducts',
        'relabel-msecvx-i50+c50-reducts',
        'relabel-msecvx-i95+c50-reducts',

    ],
    'gran-reducts-extra-params' : [
        'incl 1e-4',
        'relabel-msecvx-i50+c50-reducts',
        # 'msecvx-relab-red-cov50-i50-cer50',
        # 'msecvx-relab-red-cov50-i50-cer55',
        # 'msecvx-relab-red-cov50-i50-cer60',
        'msecvx-relab02-red-cov50-i50',
        'msecvx-relab05-red-cov50-i50',
        'msecvx-relab10-red-cov50-i50', 
        'msecvx-relab15-red-cov50-i50', 
    ],
    'gran-certainty' : [
        # 'incl 1e-4',
        'relabel-msecvx-i50+c50-reducts',
        'msecvx-relab-red-cov50-i50-cer51',
        'msecvx-relab-red-cov50-i50-cer55',
        'msecvx-relab-red-cov50-i50-cer60',
        'msecvx-relab-red-cov50-i50-cer70',
    ],
    'gran-certainty-na' : [
        # 'incl 1e-4',
        'relabel-msecvx-i50+c50-reducts',
        'msecvx-relab-red-cov50-i50-cer51-na',
        'msecvx-relab-red-cov50-i50-cer55-na',
        'msecvx-relab-red-cov50-i50-cer60-na',
        'msecvx-relab-red-cov50-i50-cer70-na',
    ],
    'gran-opt' : [
        'default-check',
        'msecvx-relab-red-cov50-i50-cer55-na',
        # 'msecvx-relab05-red-cov25-i50-cer55-na', # very bad acc
        # 'msecvx-relab05-red-cov25-i95-cer55-na', # 10% decrease
        # 'msecvx-relab05-red-cov50-i50-cer55-na', # restricted relab offers no significant advantages
        # 'msecvx-relab05-red-cov50-i95-cer55-na', # too many rules
        # 'msecvx-relab05-red-cov50-ie-6-cer55-na', # too many long rules
        # 'msecvx-relab05-red-cov55-ie-6-cer55-na', # too many long rules
        # 'msecvx-relab05-red-cov50-i55-cer55-na', # restricted relab offers no significant advantages
        # 'msecvx-relab05-red-cov50-i95-cer70-na', # awful but not a lot of rules but very long
        'msecvx-relab-red-cov50-i55-cer55-na',
        'msecvx-relab-red-cov50-i45-cer55-na'
    ],
    'gran-incl': [
        'relabel-msecvx-i50+c50-reducts',
        'gran-mse-relab-red-cov50-inc50',
        'gran-mse-relab-red-cov50',
        'gran-mse-relab-red-inc50',
        'gran-mse-relab-red',
        'default-check',
        'control'
    ],
    'owa' : [
        'incl 1e-4',
        'owa-lower-base'
    ],
    'default-check': [ # -> OK
        'incl 1e-4',
        'default-check'
    ]
}

In [272]:
nr = 'ijcrs'  # 6
save = True

In [273]:
main_folder = nr  + 'rev' + 'nocontrol' # + '-relabel'  # + "small"  # 'tables_inclusion_set2' # 'balanced_acc_incl'  # 'normal_acc'
tables_path_base = Path.cwd() / 'tables' / main_folder

In [282]:
table_acc = pd.DataFrame(index=data_sets, columns=names[nr])
for model in names[nr]:
    for data_set in data_sets:
        table_acc.loc[data_set, model] = scores[model][data_set]
table_acc.loc['mean'] = table_acc.mean(axis='rows', skipna=True)
# table_acc.drop('control', inplace=True, axis='columns')
table_acc

,control,ofrfs-1,ofrfs-0.9,ofrfs-0.8,ofrfs-0,mi-1,mi-0.9,mi-0.8,pcc-1,pcc-0.9,pcc-0.8
australian,0.824936,0.812973,0.815632,0.785569,0.582996,0.804653,0.79982,0.797998,0.805131,0.810921,0.798166
bands,0.664654,0.692661,0.690922,0.631707,0.568227,0.641887,0.654345,0.641393,0.668527,0.655481,0.631687
bupa,0.593125,0.577741,0.611038,0.611038,0.53687,0.577176,0.621101,0.602809,0.602804,0.597626,0.597626
cleveland,0.317538,0.308443,0.304744,0.267352,0.221201,0.284866,0.28655,0.296959,0.286924,0.272728,0.266336
dermatology,0.903319,0.893723,0.919663,0.909417,0.184167,0.894926,0.919807,0.8991,0.897736,0.909554,0.919966
ecoli,0.700635,0.699592,0.662471,0.662471,0.503218,0.693681,0.659115,0.674931,0.696587,0.628174,0.628174
glass,0.668254,0.655476,0.718313,0.675079,0.588869,0.672937,0.671905,0.680774,0.646389,0.675853,0.562917
heart,0.730833,0.748333,0.773333,0.6925,0.5375,0.745,0.773333,0.766667,0.7375,0.734167,0.75
ionosphere,0.910721,0.901886,0.922236,0.938122,0.839243,0.921607,0.919113,0.909899,0.906374,0.908326,0.925331
pima,0.680919,0.67193,0.659528,0.653551,0.569095,0.653351,0.630296,0.644637,0.646725,0.656687,0.644561


In [277]:
if save:
    bolded_first = table_acc.apply(lambda data: bold(data), axis=1)
    bolded_first.to_latex(buf= tables_path_base / f'tab{nr}acc.txt', escape=False)

/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_50336/22476893.py:3: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  bolded_first.to_latex(buf= tables_path_base / f'tab{nr}acc.txt', escape=False)


In [268]:
table_rule = pd.DataFrame(index=data_sets, columns=names[nr])
for model in names[nr]:
    for data_set in data_sets:
        table_rule.loc[data_set, model] = rules[model][data_set]
table_rule.loc['mean'] = table_rule.mean(axis='rows', skipna=True)
table_rule

,control,ofrfs-1,ofrfs-0.9,ofrfs-0.8,ofrfs-0,mi-1,mi-0.9,mi-0.8,pcc-1,pcc-0.9,pcc-0.8
australian,92.7,93.9,94.6,99.9,294.0,93.3,94.1,97.8,93.5,96.8,97.3
bands,56.6,56.3,56.9,59.4,228.0,58.4,61.2,58.9,60.4,63.7,66.5
bupa,76.7,76.6,79.4,79.4,165.2,76.0,88.3,86.2,79.4,86.6,86.6
cleveland,103.6,104.1,105.2,109.3,148.9,99.9,104.5,105.3,99.0,101.9,105.5
dermatology,21.0,21.0,18.4,20.7,319.5,23.4,23.8,22.5,20.5,22.1,23.3
ecoli,54.8,55.3,63.2,63.2,87.6,55.5,56.9,56.0,55.2,59.3,59.3
glass,51.8,50.9,45.4,46.4,58.9,50.6,47.0,45.9,49.8,50.0,59.4
heart,49.3,49.0,47.7,51.9,139.8,43.8,43.9,43.2,43.1,43.6,45.7
ionosphere,20.3,19.1,18.8,18.7,61.5,19.5,21.5,22.1,18.7,19.3,20.2
pima,119.1,120.8,130.5,150.4,261.1,123.8,134.3,141.3,124.7,133.2,140.4


In [269]:
if save:    
    bolded_first = table_rule.apply(lambda data: bold(data, optimum='min', format_string="%.0f"), axis=1)
    bolded_first.to_latex(buf= tables_path_base / f'tab{nr}rules.txt', escape=False)

/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_50336/1482801916.py:3: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  bolded_first.to_latex(buf= tables_path_base / f'tab{nr}rules.txt', escape=False)


In [270]:
table_attribute = pd.DataFrame(index=data_sets, columns=names[nr])
for model in names[nr]:
    for data_set in data_sets:
        table_attribute.loc[data_set, model] = attributes[model][data_set]
table_attribute.loc['mean'] = table_attribute.mean(axis='rows', skipna=True)
table_attribute

,control,ofrfs-1,ofrfs-0.9,ofrfs-0.8,ofrfs-0,mi-1,mi-0.9,mi-0.8,pcc-1,pcc-0.9,pcc-0.8
australian,5.16392,5.344019,5.326606,5.241646,1.587788,5.281809,5.002403,4.525214,5.343307,5.132113,4.490589
bands,6.446053,6.543096,6.472878,6.271234,0.574863,6.447783,6.290811,6.019594,6.187173,6.179729,5.570037
bupa,4.239517,4.23612,3.886105,3.886105,1.575835,4.208776,3.769559,3.743576,4.071409,3.642425,3.642425
cleveland,5.267326,5.279722,5.314844,5.548952,2.52409,5.620433,5.46225,5.12374,5.666346,5.634721,5.266476
dermatology,4.382869,4.377838,4.331354,4.959705,0.015624,5.175711,4.514958,4.441177,4.574146,4.651336,4.947723
ecoli,3.866034,3.848822,3.493254,3.493254,2.774694,3.858652,3.850701,3.864173,3.914932,3.762285,3.762285
glass,4.1357,4.09998,3.934342,3.809841,3.122832,4.110212,3.964584,3.780323,3.870044,3.979727,3.783036
heart,4.727108,4.724276,4.932957,5.022561,1.447553,5.149415,4.906835,4.523285,5.047044,5.170773,4.739796
ionosphere,4.490227,4.547903,4.335274,4.204327,1.861169,4.689503,4.969697,4.774836,4.663545,4.396115,4.351809
pima,5.146068,5.126304,4.753196,4.303478,2.55821,5.058648,4.823556,4.403376,5.118021,4.819491,4.441028


In [271]:
if save:
    bolded_first = table_attribute.apply(lambda data: bold(data, optimum='min', format_string="%.2f"), axis=1)
    bolded_first.to_latex(buf= tables_path_base / f'tab{nr}atts.txt', escape=False)

/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_50336/357892810.py:3: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  bolded_first.to_latex(buf= tables_path_base / f'tab{nr}atts.txt', escape=False)


## Statistical testing

In [283]:
from scipy import stats
import scikit_posthocs as sph

In [299]:
subject = table_attribute

In [300]:
no_mean =  subject.drop(labels = 'mean', axis = 'index')
# no_mean.drop('max', axis='columns', inplace=True)

In [301]:
# no_mean

In [308]:
stats.wilcoxon(no_mean['ofrfs-0.9'],no_mean['control'], alternative='less')

WilcoxonResult(statistic=31.0, pvalue=0.007965087890625)

In [304]:
f_test = stats.friedmanchisquare(*no_mean.values)
f_test

FriedmanchisquareResult(statistic=148.21212121212125, pvalue=5.476276163239811e-23)

In [305]:
post_hocs = sph.posthoc_conover_friedman(no_mean.values, p_adjust='Holm')
post_hocs.columns=no_mean.columns
post_hocs.index=no_mean.columns
post_hocs

,control,ofrfs-1,ofrfs-0.9,ofrfs-0.8,ofrfs-0,mi-1,mi-0.9,mi-0.8,pcc-1,pcc-0.9,pcc-0.8
control,1.000000e+00,1.000000e+00,1.000000,0.100073,1.314831e-07,1.000000e+00,1.000000,0.019170,1.000000e+00,1.000000,0.030773
ofrfs-1,1.000000e+00,1.000000e+00,1.000000,0.190211,4.521138e-07,1.000000e+00,1.000000,0.041091,1.000000e+00,1.000000,0.063892
ofrfs-0.9,1.000000e+00,1.000000e+00,1.000000,1.000000,1.474104e-03,1.738864e-01,1.000000,1.000000,1.000000e+00,1.000000,1.000000
ofrfs-0.8,1.000733e-01,1.902114e-01,1.000000,1.000000,5.573215e-02,6.617121e-03,1.000000,1.000000,1.000733e-01,1.000000,1.000000
ofrfs-0,1.314831e-07,4.521138e-07,0.001474,0.055732,1.000000e+00,1.435972e-09,0.000052,0.246245,1.314831e-07,0.000192,0.173886
mi-1,1.000000e+00,1.000000e+00,0.173886,0.006617,1.435972e-09,1.000000e+00,1.000000,0.000820,1.000000e+00,0.615763,0.001474
mi-0.9,1.000000e+00,1.000000e+00,1.000000,1.000000,5.231247e-05,1.000000e+00,1.000000,0.615763,1.000000e+00,1.000000,0.841416
mi-0.8,1.916961e-02,4.109147e-02,1.000000,1.000000,2.462446e-01,8.204504e-04,0.615763,1.000000,1.916961e-02,1.000000,1.000000
pcc-1,1.000000e+00,1.000000e+00,1.000000,0.100073,1.314831e-07,1.000000e+00,1.000000,0.019170,1.000000e+00,1.000000,0.030773
pcc-0.9,1.000000e+00,1.000000e+00,1.000000,1.000000,1.922273e-04,6.157634e-01,1.000000,1.000000,1.000000e+00,1.000000,1.000000


In [189]:
best = no_mean.max(axis='columns')
dif = no_mean.subtract(best, axis='rows').abs()
dif.max(axis='rows').sort_values()

ofrfs-0        0.0
ofrfs-0.8    374.0
ofrfs-0.9    376.4
ofrfs-1      378.0
control      379.8
dtype: object

In [188]:
ranks = no_mean.rank(axis='columns', ascending=False)
friedman_ranks = ranks.mean()
print(friedman_ranks.sort_values().to_latex(escape=False))

\begin{tabular}{lr}
\toprule
{} &         0 \\
\midrule
ofrfs-0   &  1.000000 \\
ofrfs-0.8 &  2.666667 \\
ofrfs-0.9 &  3.361111 \\
control   &  3.972222 \\
ofrfs-1   &  4.000000 \\
\bottomrule
\end{tabular}



/var/folders/_f/yf3dkr5x1397v4ntwhncxfhw0000gn/T/ipykernel_50336/2662355144.py:3: FutureWarning: In future versions `DataFrame.to_latex` is expected to utilise the base implementation of `Styler.to_latex` for formatting and rendering. The arguments signature may therefore change. It is recommended instead to use `DataFrame.style.to_latex` which also contains additional functionality.
  print(friedman_ranks.sort_values().to_latex(escape=False))


In [67]:
ranks['lower-new-check'].value_counts()

KeyError: 'lower-new-check'

In [68]:
ranks.apply(pd.value_counts)

,incl 1e-6,incl 1e-4,incl 1e-3,incl 1e-2,cv-thres
1.0,3,5.0,1.0,3,2
2.0,4,3.0,3.0,1,2
2.5,1,NaN,1.0,1,1
3.0,5,1.0,7.0,1,2
4.0,1,1.0,3.0,3,5
5.0,1,5.0,NaN,6,3


In [188]:
median = no_mean.median(axis='rows')
median

incl 1e-6    0.680919
incl 1e-4    0.687633
incl 1e-3    0.676793
incl 1e-2    0.660827
cv-thres     0.668552
dtype: float64

On high IR datasets (>6)

In [196]:
ir_datasets = [
    'cleveland',
    'ecoli',
    'glass',
    'spectfheart',
    'winequality-red',
    'yeast'
]

ir_no_mean = no_mean.loc[ir_datasets]
# ir_no_mean.drop('qr', axis='columns', inplace=True)

In [197]:
f_test = stats.friedmanchisquare(*ir_no_mean.values)
f_test

FriedmanchisquareResult(statistic=14.714285714285708, pvalue=0.011655524770004939)

In [198]:
post_hocs = sph.posthoc_conover_friedman(ir_no_mean.values)
post_hocs.columns=ir_no_mean.columns
post_hocs.index=ir_no_mean.columns
post_hocs

,lower-new-check,modlem,qr,furia,ripper
lower-new-check,1.000000,0.233052,0.004900,0.094249,0.094249
modlem,0.233052,1.000000,0.067585,0.603959,0.603959
qr,0.004900,0.067585,1.000000,0.175230,0.175230
furia,0.094249,0.603959,0.175230,1.000000,1.000000
ripper,0.094249,0.603959,0.175230,1.000000,1.000000


In [199]:
stats.wilcoxon(ir_no_mean['lower-new-check'], ir_no_mean['ripper'], alternative="greater")

WilcoxonResult(statistic=18.0, pvalue=0.078125)

In [200]:
wil_ph = sph.posthoc_nemenyi_friedman(ir_no_mean.values)
wil_ph.columns=ir_no_mean.columns
wil_ph.index=ir_no_mean.columns
wil_ph

,lower-new-check,modlem,qr,furia,ripper
lower-new-check,1.000000,0.680037,0.008987,0.359170,0.359170
modlem,0.680037,1.000000,0.261866,0.900000,0.900000
qr,0.008987,0.261866,1.000000,0.576422,0.576422
furia,0.359170,0.900000,0.576422,1.000000,0.900000
ripper,0.359170,0.900000,0.576422,0.900000,1.000000


0.03